# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [1]:
# Standard library imports for this phase
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")


In [2]:
DATA_PATH = Path("..") / "data" / "raw" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


Loaded dataset: 1470 rows x 35 columns


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:
columns_to_drop = ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"]
columns_to_keep = [col for col in df.columns if col not in columns_to_drop]

selection_rationale = {
    "dropped_columns": {
        "EmployeeNumber": "Identifier only; not predictive in a meaningful business sense.",
        "EmployeeCount": "Constant value for every record.",
        "Over18": "Constant value for every record.",
        "StandardHours": "Constant value for every record.",
    },
    "kept_columns_count": len(columns_to_keep),
}

df_selected = df[columns_to_keep].copy()

selection_rationale


{'dropped_columns': {'EmployeeNumber': 'Identifier only; not predictive in a meaningful business sense.',
  'EmployeeCount': 'Constant value for every record.',
  'Over18': 'Constant value for every record.',
  'StandardHours': 'Constant value for every record.'},
 'kept_columns_count': 31}

In [4]:
df_selected = df_selected[df_selected["Attrition"].notna()].copy()
print(f"Shape after row filtering: {df_selected.shape}")


Shape after row filtering: (1470, 31)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [5]:
df_clean = df_selected.copy()

for col in df_clean.select_dtypes(include="object").columns:
    df_clean[col] = df_clean[col].str.strip()

for col in df_clean.select_dtypes(include=np.number).columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

missing_after_type_fix = df_clean.isna().sum().sum()
print(f"Missing values after type standardisation: {missing_after_type_fix}")


Missing values after type standardisation: 0


In [6]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = len(df_clean)

print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")


Removed 0 duplicate rows. Remaining: 1470 rows.


In [7]:
outlier_columns = [
    "MonthlyIncome",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager",
    "DistanceFromHome",
    "NumCompaniesWorked",
]

outlier_caps = {}
for col in outlier_columns:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
    outlier_caps[col] = {"lower_cap": round(lower, 2), "upper_cap": round(upper, 2)}

pd.DataFrame(outlier_caps).T


,lower_cap,upper_cap
MonthlyIncome,-5291.0,16581.0
TotalWorkingYears,-7.5,28.5
YearsAtCompany,-6.0,18.0
YearsInCurrentRole,-5.5,14.5
YearsSinceLastPromotion,-4.5,7.5
YearsWithCurrManager,-5.5,14.5
DistanceFromHome,-16.0,32.0
NumCompaniesWorked,-3.5,8.5


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [8]:
df_clean["Attrition_Flag"] = df_clean["Attrition"].eq("Yes").astype(int)
df_clean["IncomePerJobLevel"] = df_clean["MonthlyIncome"] / df_clean["JobLevel"].replace(0, np.nan)
df_clean["TenureRatio"] = df_clean["YearsInCurrentRole"] / (df_clean["YearsAtCompany"] + 1)
df_clean["YearsSincePromotionRatio"] = df_clean["YearsSinceLastPromotion"] / (df_clean["YearsAtCompany"] + 1)
df_clean["IsFrequentTraveler"] = df_clean["BusinessTravel"].eq("Travel_Frequently").astype(int)
df_clean["HasLongCommute"] = (df_clean["DistanceFromHome"] >= 20).astype(int)

engineered_features = [
    "Attrition_Flag",
    "IncomePerJobLevel",
    "TenureRatio",
    "YearsSincePromotionRatio",
    "IsFrequentTraveler",
    "HasLongCommute",
]

df_clean[engineered_features].head()


,Attrition_Flag,IncomePerJobLevel,TenureRatio,YearsSincePromotionRatio,IsFrequentTraveler,HasLongCommute
0,1,2996.5,0.571429,0.000000,0,0
1,0,2565.0,0.636364,0.090909,1,0
2,1,2090.0,0.000000,0.000000,0,0
3,0,2909.0,0.777778,0.333333,1,0
4,0,3468.0,0.666667,0.666667,0,0


In [9]:
categorical_cols = df_clean.select_dtypes(include="object").columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "Attrition"]

df_encoded = pd.get_dummies(
    df_clean.drop(columns=["Attrition"]),
    columns=categorical_cols,
    drop_first=True,
)

print(f"Shape after encoding: {df_encoded.shape}")
df_encoded.head()


Shape after encoding: (1470, 50)


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition_Flag,IncomePerJobLevel,TenureRatio,YearsSincePromotionRatio,IsFrequentTraveler,HasLongCommute,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Male,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,41,1102,1,2,2,94,3,2,4,5993,19479,8.0,11,3,1,0,8.0,0,1,6,4.0,0.0,5.0,1,2996.5,0.571429,0.000000,0,0,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,True
1,49,279,8,1,3,61,2,2,2,5130,24907,1.0,23,4,4,1,10.0,3,3,10,7.0,1.0,7.0,0,2565.0,0.636364,0.090909,1,0,True,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False
2,37,1373,2,2,4,92,2,1,3,2090,2396,6.0,15,3,2,0,7.0,3,3,0,0.0,0.0,0.0,1,2090.0,0.000000,0.000000,0,0,False,True,True,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False,True,True
3,33,1392,3,4,4,56,3,1,3,2909,23159,1.0,11,3,3,0,8.0,3,3,8,7.0,3.0,0.0,0,2909.0,0.777778,0.333333,1,0,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,True
4,27,591,2,1,1,40,3,1,2,3468,16632,8.5,12,3,4,1,6.0,3,3,2,2.0,2.0,2.0,0,3468.0,0.666667,0.666667,0,0,False,True,True,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,True,False,False


In [10]:
scaling_decision = {
    "applied_in_phase_3": False,
    "reason": (
        "Scaling is deferred to the modelling pipeline in Phase 4 to avoid "
        "data leakage and to keep the prepared dataset reusable across models."
    ),
}

scaling_decision


{'applied_in_phase_3': False,
 'reason': 'Scaling is deferred to the modelling pipeline in Phase 4 to avoid data leakage and to keep the prepared dataset reusable across models.'}

---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [11]:
integration_notes = {
    "multiple_sources_used": False,
    "decision": "No merge step required because the project uses a single CSV source.",
}

df_integrated = df_encoded.copy()
integration_notes


{'multiple_sources_used': False,
 'decision': 'No merge step required because the project uses a single CSV source.'}

In [12]:
df_integrated.head()


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition_Flag,IncomePerJobLevel,TenureRatio,YearsSincePromotionRatio,IsFrequentTraveler,HasLongCommute,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Male,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,41,1102,1,2,2,94,3,2,4,5993,19479,8.0,11,3,1,0,8.0,0,1,6,4.0,0.0,5.0,1,2996.5,0.571429,0.000000,0,0,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,True
1,49,279,8,1,3,61,2,2,2,5130,24907,1.0,23,4,4,1,10.0,3,3,10,7.0,1.0,7.0,0,2565.0,0.636364,0.090909,1,0,True,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False
2,37,1373,2,2,4,92,2,1,3,2090,2396,6.0,15,3,2,0,7.0,3,3,0,0.0,0.0,0.0,1,2090.0,0.000000,0.000000,0,0,False,True,True,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False,True,True
3,33,1392,3,4,4,56,3,1,3,2909,23159,1.0,11,3,3,0,8.0,3,3,8,7.0,3.0,0.0,0,2909.0,0.777778,0.333333,1,0,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,True
4,27,591,2,1,1,40,3,1,2,3468,16632,8.5,12,3,4,1,6.0,3,3,2,2.0,2.0,2.0,0,3468.0,0.666667,0.666667,0,0,False,True,True,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,True,False,False


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [13]:
ordered_columns = ["Attrition_Flag"] + [col for col in df_integrated.columns if col != "Attrition_Flag"]
df_final = df_integrated[ordered_columns].copy()

df_final.columns = [col.replace(" ", "_").replace("&", "and") for col in df_final.columns]

formatting_summary = {
    "target_column": "Attrition_Flag",
    "final_column_count": len(df_final.columns),
    "row_count": len(df_final),
}

formatting_summary


{'target_column': 'Attrition_Flag',
 'final_column_count': 50,
 'row_count': 1470}

In [14]:
print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_final.shape}")
print(f"Missing values: {int(df_final.isna().sum().sum())}")
print(f"Duplicate rows: {int(df_final.duplicated().sum())}")

df_final.head()


FINAL PREPARED DATASET SUMMARY
Shape: (1470, 50)
Missing values: 0
Duplicate rows: 0


,Attrition_Flag,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,IncomePerJobLevel,TenureRatio,YearsSincePromotionRatio,IsFrequentTraveler,HasLongCommute,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research_and_Development,Department_Sales,EducationField_Life_Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical_Degree,Gender_Male,JobRole_Human_Resources,JobRole_Laboratory_Technician,JobRole_Manager,JobRole_Manufacturing_Director,JobRole_Research_Director,JobRole_Research_Scientist,JobRole_Sales_Executive,JobRole_Sales_Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes
0,1,41,1102,1,2,2,94,3,2,4,5993,19479,8.0,11,3,1,0,8.0,0,1,6,4.0,0.0,5.0,2996.5,0.571429,0.000000,0,0,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,True
1,0,49,279,8,1,3,61,2,2,2,5130,24907,1.0,23,4,4,1,10.0,3,3,10,7.0,1.0,7.0,2565.0,0.636364,0.090909,1,0,True,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False
2,1,37,1373,2,2,4,92,2,1,3,2090,2396,6.0,15,3,2,0,7.0,3,3,0,0.0,0.0,0.0,2090.0,0.000000,0.000000,0,0,False,True,True,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False,True,True
3,0,33,1392,3,4,4,56,3,1,3,2909,23159,1.0,11,3,3,0,8.0,3,3,8,7.0,3.0,0.0,2909.0,0.777778,0.333333,1,0,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,True
4,0,27,591,2,1,1,40,3,1,2,3468,16632,8.5,12,3,4,1,6.0,3,3,2,2.0,2.0,2.0,3468.0,0.666667,0.666667,0,0,False,True,True,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,True,False,False


In [15]:
OUTPUT_DIR = Path("..") / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "hr_attrition_prepared.csv"
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Prepared dataset saved to: {OUTPUT_PATH.resolve()}")


Prepared dataset saved to: C:\vscode\workspace\ds\CRISP-DM Guiding Notebooks-20260415\data\processed\hr_attrition_prepared.csv
